In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch import nn, optim
from torch.utils.data import DataLoader
import torchvision.datasets as datasets

# Define CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )

    def forward(self, x):
        return self.fc(self.conv(x))

# Define progressive augmentation scheduler
def get_transform(epoch):
    if epoch < 10:
        print("Progressive Augmentation >>> Case 1")
        return transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor()
        ])
    elif epoch < 20:
        print("Progressive Augmentation >>> Case 2")
        return transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor()
        ])
    else:
        print("Progressive Augmentation >>> Case 3")
        return transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
            transforms.RandomRotation(15),
            transforms.RandomErasing(p=0.2),
            transforms.ToTensor()
        ])





In [6]:
# Parameter Set Up
model = SimpleCNN()

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

total_epochs = 20

# Training Model
for epoch in range(total_epochs):

  # Progressive Transform
  transform = get_transform(epoch)

  # Dataset with Progressive Transform
  trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
  trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

  model.train()
  total_loss = 0

  for images, labels in trainloader:
    images, labels = images.to(device), labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()


  # แสดงค่า learning rate และ loss
  current_lr = optimizer.param_groups[0]['lr']
  print(f"Epoch {epoch+1}: LR = {current_lr:.6f}, Loss = {total_loss:.4f}")


Progressive Augmentation >>> Case 1
Epoch 1: LR = 0.001000, Loss = 1246.3496
Progressive Augmentation >>> Case 1
Epoch 2: LR = 0.001000, Loss = 1015.6079
Progressive Augmentation >>> Case 1
Epoch 3: LR = 0.001000, Loss = 924.2589
Progressive Augmentation >>> Case 1
Epoch 4: LR = 0.001000, Loss = 849.6002
Progressive Augmentation >>> Case 1
Epoch 5: LR = 0.001000, Loss = 800.0217
Progressive Augmentation >>> Case 1
Epoch 6: LR = 0.001000, Loss = 756.4806
Progressive Augmentation >>> Case 1
Epoch 7: LR = 0.001000, Loss = 723.6660
Progressive Augmentation >>> Case 1
Epoch 8: LR = 0.001000, Loss = 696.9016
Progressive Augmentation >>> Case 1
Epoch 9: LR = 0.001000, Loss = 673.8261
Progressive Augmentation >>> Case 1
Epoch 10: LR = 0.001000, Loss = 652.9446
Progressive Augmentation >>> Case 2
Epoch 11: LR = 0.001000, Loss = 651.8515
Progressive Augmentation >>> Case 2
Epoch 12: LR = 0.001000, Loss = 637.8007
Progressive Augmentation >>> Case 2
Epoch 13: LR = 0.001000, Loss = 621.3867
Progre